### IMPORT NECESSARY LIBRARIES

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np

### Load dataset and see structure

In [ ]:
district_df = pd.read_csv("./datasets/nepal_district_centroids.csv")
dengue_df = pd.read_csv("./datasets/dengu_temporal.csv")

print("DENGUE DF DESCRIPTION")
display(dengue_df.describe())

print("\nDENGUE DF DATA STRUCTURE")
display(dengue_df.head(3))

print("\nDISTRICT DATA STRUCTURE")
display(district_df.head())

### Count NULL Values in DENGHE DF

In [ ]:
dengue_df.isnull().sum()

### Count Values in S_res and T_res
Exactly see on what the most data is distributed in both spatial and temporal. For further extraction

In [ ]:
print("S_res counts:")
display(dengue_df["S_res"].value_counts())

print("\nT_res counts:")
display(dengue_df["T_res"].value_counts())

### Select row with only S_res Admin2 (District) and T_res Month

In [ ]:
dengue_df = dengue_df[dengue_df["S_res"]=="Admin2"]
dengue_df = dengue_df[dengue_df["T_res"]=="Month"]

# SHOW WHAT ADMIN 2 CONTAINS
print("Admin2 Counts")
dengue_df["adm_2_name"].value_counts()

### Create District, Start Date, End Date and Month in clean way and Remove unnecessary columns
Overall clean data of dengue_df

In [ ]:
# Create Month
dengue_df["district"] = dengue_df["adm_2_name"].fillna(dengue_df["full_name"].str.split(",").str[-1])

# Create Date
dengue_df["start_date"] = pd.to_datetime(dengue_df["calendar_start_date"])
dengue_df["end_date"] = pd.to_datetime(dengue_df["calendar_end_date"])
dengue_df["year"] = dengue_df["Year"]
dengue_df["month"] = dengue_df["start_date"].dt.month

# Remove columns
dengue_df.drop(columns=[
    "adm_0_name",
    "adm_1_name",
    "adm_2_name",
    "full_name",
    "ISO_A0",
    "FAO_GAUL_code",
    "RNE_iso_code",
    "IBGE_code",
    "calendar_start_date",
    "calendar_end_date",
    "case_definition_standardised",
    "S_res",
    "T_res",
    "UUID",
    "region"
], inplace=True)

# Show the head
dengue_df.head(5)

### MERGE WITH DISTRICT DF

In [ ]:
# strip if additional spaces
dengue_df["district"] = dengue_df["district"].str.strip().str.lower()
district_df["DISTRICT"] = district_df["DISTRICT"].str.strip().str.lower()

dengue_df = pd.merge(dengue_df, district_df, how="inner", left_on="district", right_on="DISTRICT")

# rename and remove
dengue_df["latitude"] = dengue_df["LATITUDE"]
dengue_df["longitude"] = dengue_df["LONGITUDE"]
dengue_df["year"] = dengue_df["Year"]

dengue_df.drop(columns=["LATITUDE", "LONGITUDE", "DISTRICT", "Year"], inplace=True)

# see all null values
print("ALL NULL VALS")
display(dengue_df.isnull().sum())

# Show head
dengue_df.head()

### Check Year And Month Distributions

In [ ]:
# Count Years
print("YEAR COUNTS")
display(dengue_df["year"].value_counts())

print("MONTH COUNTS")
display(dengue_df["month"].value_counts())

print("DISTRICT COUNTS")
display(dengue_df["district"].value_counts())

print(f"TOTAL DATA LENGTH = {len(dengue_df)}")

### Fetching Temporal Weather data

In [ ]:
# Range Of Date
date_from = "2021-10-01"
date_to = "2025-01-01"

# set of latitude and longitude
unique_locs = list(set(zip(
        dengue_df["latitude"].to_list(), 
        dengue_df["longitude"].to_list()
        )
    ))

print(f"LENGTH OF LONGITUDE LATITUDE PAIRS = {len(unique_locs)}")

### Code from Open meteo itserf

In [ ]:
import numpy as np
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup Open-Meteo API client with caching and retries
cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"

# Split into separate lists for batch API requests
latitudes = [loc[0] for loc in unique_locs]
longitudes = [loc[1] for loc in unique_locs]

params = {
    "latitude": latitudes,
    "longitude": longitudes,
    "start_date": date_from,
    "end_date": date_to,
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_min",
        "precipitation_sum",
        "relative_humidity_2m_mean",
        "soil_moisture_0_to_7cm_mean",
        "daylight_duration",
    ],
    "timezone": "auto",
}

# Fetch data for all locations at once
responses = openmeteo.weather_api(url, params=params)

all_location_dfs = []

# Loop over responses AND original input coordinate pairs together
for response, (orig_lat, orig_lon) in zip(responses, unique_locs):

    daily = response.Daily()

    # Extract NumPy arrays for daily metrics
    daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
    daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
    daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
    daily_relative_humidity_2m_mean = daily.Variables(3).ValuesAsNumpy()
    daily_soil_moisture_0_to_7cm_mean = daily.Variables(4).ValuesAsNumpy()
    daily_daylight_duration = daily.Variables(5).ValuesAsNumpy()

    # Build local time series range
    date_range = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left",
    ).tz_convert(response.Timezone().decode())

    # Build DataFrame using EXACT original latitude/longitude from dengue_df
    daily_data = {
        "date": date_range,
        "latitude": orig_lat,  # <--- Fix: Use exact input coordinate
        "longitude": orig_lon,  # <--- Fix: Use exact input coordinate
        "temperature_2m_mean": daily_temperature_2m_mean,
        "temperature_2m_min": daily_temperature_2m_min,
        "precipitation_sum": daily_precipitation_sum,
        "relative_humidity_2m_mean": daily_relative_humidity_2m_mean,
        "soil_moisture_0_to_7cm_mean": daily_soil_moisture_0_to_7cm_mean,
        "daylight_duration": daily_daylight_duration,
    }

    df_loc = pd.DataFrame(data=daily_data)

    # Add Year and Month columns
    df_loc["year"] = df_loc["date"].dt.year
    df_loc["month"] = df_loc["date"].dt.month

    all_location_dfs.append(df_loc)

# Combine all individual DataFrames into one large DataFrame
weather_df = pd.concat(all_location_dfs, ignore_index=True)

print("\nFetched Final Weather DataFrame:\n")
weather_df.head()

### Finding Average value for each month for each District

In [ ]:
# Define how each variable should be aggregated to monthly
agg_rules = {
    "temperature_2m_mean": "mean",
    "temperature_2m_min": "mean",
    "precipitation_sum": "sum",  # Total rain for the month
    "relative_humidity_2m_mean": "mean",
    "soil_moisture_0_to_7cm_mean": "mean",
    "daylight_duration": "mean",
}

# Group by location, year, and month
monthly_weather_df = (
    weather_df.groupby(["latitude", "longitude", "year", "month"])
    .agg(agg_rules)
    .reset_index()
)

# Rename columns to clearly reflect monthly totals vs means
monthly_weather_df = monthly_weather_df.rename(
    columns={
        "precipitation_sum": "precipitation_monthly_sum",
        "temperature_2m_mean": "temperature_2m_mean_avg",
        "temperature_2m_min": "temperature_2m_min_avg",
        "relative_humidity_2m_mean": "relative_humidity_2m_mean_avg",
        "soil_moisture_0_to_7cm_mean": "soil_moisture_0_to_7cm_mean_avg",
        "daylight_duration": "daylight_duration_avg",
    }
)

# Display the result
display(monthly_weather_df.head())

### Find Lagged Month data from past 2 months

In [ ]:
# 1. Sort monthly weather chronologically per location
monthly_weather_df = monthly_weather_df.sort_values(
    by=["latitude", "longitude", "year", "month"]
).reset_index(drop=True)

weather_vars = [
    "temperature_2m_mean_avg",
    "temperature_2m_min_avg",
    "precipitation_monthly_sum",
    "relative_humidity_2m_mean_avg",
    "soil_moisture_0_to_7cm_mean_avg",
    "daylight_duration_avg",
]

# 2. Compute 1m and 2m lags on weather dataframe directly
grouped_weather = monthly_weather_df.groupby(["latitude", "longitude"])

for var in weather_vars:
    monthly_weather_df[f"{var}_lag_1m"] = grouped_weather[var].shift(1)
    monthly_weather_df[f"{var}_lag_2m"] = grouped_weather[var].shift(2)

# 3. Direct merge on exact latitude & longitude
final_model_df = pd.merge(
    dengue_df,
    monthly_weather_df,
    on=["latitude", "longitude", "year", "month"],
    how="inner",
)

# 4. Drop rows missing lag features (Oct & Nov 2021)
lag_cols = [f"{v}_lag_2m" for v in weather_vars]
final_model_df = final_model_df.dropna(subset=lag_cols).reset_index(drop=True)

# 5. Clean up temporary columns
dropables = ["start_date", "end_date"]
final_model_df.drop(
    columns=[c for c in dropables if c in final_model_df.columns], inplace=True
)

print(f"Final Dataset Shape: {final_model_df.shape}")
final_model_df.head()

### See all columns, shape and over all data

In [ ]:
print("ALL COLUMNS:")
print(final_model_df.columns.to_list())

print(f"\nDENGUE SHAPE = {dengue_df.shape}")
print(f"\nFINAL SHAPE = {final_model_df.shape}")

print("\nNULL VALUES")
print(final_model_df.isnull().sum())

In [ ]:
# DROP DATA
dropables = ['start_date', 'end_date', 'latitude', 'longitude', 'lat_round', 'lon_round']

# errors='ignore' safely skips missing columns without throwing a KeyError
final_model_df.drop(columns=dropables, errors="ignore", inplace=True)

print("Final Dataset")
final_model_df.head()

In [ ]:
final_model_df.columns

### Randomize save and split save

In [ ]:
final_model_df = final_model_df.sample(frac=1, random_state=41)
final_model_df.to_csv("./datasets/dengue_final_dataset.csv", index=False)

In [ ]:
train_size = int(0.65 * len(final_model_df))
val_size = train_size + int(0.20 * len(final_model_df))

train_set = final_model_df.iloc[:train_size]
val_set = final_model_df.iloc[train_size:val_size]
test_set = final_model_df.iloc[val_size:]

print(f"TRAIN LEN = {len(train_set)}")
print(f"VAL LEN = {len(val_set)}")
print(f"TEST LEN = {len(test_set)}")


In [ ]:
train_set.to_csv("./datasets/train.csv", index=False)
train_set.head()

In [ ]:
val_set.to_csv("./datasets/val.csv", index=False)
val_set.head()

In [ ]:
test_set.to_csv("./datasets/test.csv", index=False)
test_set.head()